# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data:** paired spatial-transcriptomics slides (Zhuang ABCA mouse-brain atlas).

A runnable companion to the `paired_slides_eval` suite. The notebook follows one linear workflow:

1. **Generate cells to evaluate** — confirm or regenerate per-model `.h5ad` artifacts.
2. **Build source, target, and classifier slide pickles** in one shared whitened `X_pca(50)` basis.
3. **Train the classifier probes** used by `ct/*` and `recon/*`.
4. **Train OT-CFM on the shared basis** — export `shared_pca.npz` from `abca_pair.pkl`, then train OT-CFM against it.
5. **Train NicheFlow VFM and CFM** — keep NicheFlow in the same pair preprocessing frame.
6. **Run evaluation, tables, and UMAP** from the shared target frame.

Everything is scored through the **same** apparatus. OT-CFM emits flat cells in the shared PCA coordinate convention, while NicheFlow emits niche-shaped cells with fixed target-centroid ground truth. During classifier/reconstruction evaluation, flat OT-CFM slides are now paired to the same fixed target centroids by OT assignment in the standardised coordinate frame. All paths below are **relative to `notebooks/`**.


## 0. Environment setup

Per-project `uv` venv (run from the `eval_metrics/` repo root):


In [ ]:
%%bash
UV_LINK_MODE=copy uv sync --extra pipeline --extra nicheflow --extra classifier 


In [ ]:
%%bash
cd /work/magroup/shared/enfuzers/generative_modeling/eval_metrics
# env -u VIRTUAL_ENV uv run python -m ipykernel install --user \
#     --name eval_metrics --display-name "Python (eval_metrics)"


In [ ]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.version.cuda)


In [ ]:
import importlib

import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))


## 1. Generate cells to evaluate

The metrics compare the real **target slide** against a model's **generated cells**. This section is for checking or regenerating the `.h5ad` artifacts that evaluation consumes.

| key | emits | eval-time pairing |
|---|---|---|
| `otcfm` / `otcfm_spatial` | flat shared-PCA cells + 2D coords | fixed target centroids matched to generated cells by OT |
| `nicheflow_cfm` / `nicheflow_vfm` | niche-shaped shared-PCA cells + 2D coords | model-supplied `gt_x/gt_pos/gt_ct` at fixed target centroids |

The first cell checks the files. The following optional cells regenerate artifacts from existing checkpoints; run the relevant training sections first if you need new checkpoints.


In [ ]:
import os

# Each model's generated cells (one flat/niche .h5ad per model).
ARTIFACTS = {
    "otcfm":          "../artifacts/otcfm_shared50_1025/generated.h5ad",               
    "otcfm_spatial":  "../artifacts/otcfm_spatial_shared50_1025/generated.h5ad",       
    "nicheflow_cfm":  "../artifacts/nicheflow_cfm_unaligned/generated.h5ad",  
    "nicheflow_vfm":  "../artifacts/nicheflow_vfm_unaligned/generated.h5ad",           
}
for name, path in ARTIFACTS.items():
    print(f"{'OK     ' if os.path.exists(path) else 'MISSING'}  {name:14s} {path}")


In [ ]:
%%bash
# OPTIONAL — (re)generate NicheFlow CFM cells on the allocated GPU (overwrites the artifact; needs a
# checkpoint).
cd ..
# uv run python -m paired_slides_eval.generate generator=nicheflow \
#     source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
#     target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
#     checkpoint=ckpts/NicheFlow_CFM_ABCA.ckpt \
#     generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad
../nicheflow_mba/.venv/bin/python -m paired_slides_eval.generate generator=nicheflow \
      source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
      target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
      checkpoint=../nicheflow_mba/ckpts/NicheFlow_CFM_ABCA.ckpt \
      generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad


In [ ]:
%%bash
# Generate Nicheflow VFM cells 
cd ..
../nicheflow_mba/.venv/bin/python -m paired_slides_eval.generate generator=nicheflow \
      generator.variant=vfm \
      source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
      target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
      checkpoint=../nicheflow_mba/ckpts/NicheFlow_VFM_ABCA.ckpt \
      generated_out=artifacts/nicheflow_vfm_unaligned/generated.h5ad


In [ ]:
%%bash
# OPTIONAL — (re)generate the OT-CFM cells from checkpoints trained in §4.
# The generator keeps expression in the model's shared whitened-PCA space; evaluation then
# auto-detects that the width already matches the target PCA basis and passes it through.
cd ..
FM=../fm_mnist
DATA=../nicheflow_mba/data

uv run python -m paired_slides_eval.generate generator=otcfm \
    target=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    source=null \
    checkpoint=$FM/outputs/cfm_shared50_1025/ckpt_final.pt \
    generator.fm_root=$FM \
    generated_out=artifacts/otcfm_shared50_1025/generated.h5ad

uv run python -m paired_slides_eval.generate generator=otcfm_spatial \
    source=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    target=$DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    checkpoint=$FM/outputs/cfm_spatial_shared50_1025/ckpt_final.pt \
    generator.fm_root=$FM \
    generated_out=artifacts/otcfm_spatial_shared50_1025/generated.h5ad


## 2. Build the source, target, and classifier slide pickles

One `preprocess_pair` fit defines the shared measuring apparatus and is reused everywhere:

- **Expression:** `normalize_total → log1p → shared PCA (fit on source+target) → whiten`.
- **Coordinates:** per-slide standardisation `(pos − mean) / std`.
- **Fixed target centroids:** `subsampled_timepoint_idx` is persisted and later used to pair flat OT-CFM slides to the same target centroid grid as NicheFlow.

| file | what it is |
|---|---|
| `../data/abca_pair.pkl` | the apparatus **and** the eval target — source `1.024` → target `1.025` |
| `../data/abca_1.026_clf.pkl` | the held-out **classifier** slide `1.026` projected into the *same* basis |

Slide identities: **target = `1.025`**, **generation pair = `1.024 → 1.025`**, **classifier slide = `1.026`**.


In [ ]:
%%bash
uv run python -m paired_slides_eval.adapters.prepare_shared_slides \
  --source ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
  --target ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
  --classifier_slide ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.026.h5ad \
  --ct_key class --n_pcs 50 \
  --out_pair ../data/abca_pair.pkl --out_classifier ../data/abca_1.026_clf.pkl


In [ ]:
import pickle

ds = pickle.load(open("../data/abca_pair.pkl", "rb"))
print("shared whitened X_pca dim =", ds.X_pca.shape[1], "(the basis every model is scored in)")
print("target slide =", ds.timepoints_ordered[-1], "| cells =", ds.X_pca.shape[0])


In [ ]:
import pickle

from paired_slides_eval.adapters.nicheflow.preprocess import (
    preprocess_classifier_slide, preprocess_pair,
)

D = "../../nicheflow_mba/data"
SOURCE_H5AD     = f"{D}/adata_Zhuang_Zhuang-ABCA-1.024.h5ad"   # source (A)
TARGET_H5AD     = f"{D}/adata_Zhuang_Zhuang-ABCA-1.025.h5ad"   # target (B) = eval target
CLASSIFIER_H5AD = f"{D}/adata_Zhuang_Zhuang-ABCA-1.026.h5ad"   # held-out classifier slide
PAIR_PKL, CLF_PKL = "../data/abca_pair.pkl", "../data/abca_1.026_clf.pkl"
N_PCS, CT_KEY = 50, "class"

# In-process equivalent of the §2 CLI: fit the shared source+target PCA once; both the pair pickle
# (eval target) and the classifier slide land in that one whitened X_pca(50) basis.
ds_pair, pre = preprocess_pair(SOURCE_H5AD, TARGET_H5AD, n_pcs=N_PCS, cell_type_column=CT_KEY)
clf_ds = preprocess_classifier_slide(CLASSIFIER_H5AD, pre, cell_type_column=CT_KEY)
pickle.dump(ds_pair, open(PAIR_PKL, "wb"))
pickle.dump(clf_ds, open(CLF_PKL, "wb"))
print(f"built {PAIR_PKL} (X_pca dim={ds_pair.X_pca.shape[1]}) and {CLF_PKL} "
      f"({len(clf_ds.ct_ordered)} classes)")


## 3. Train the classifier probes

The label-free metrics (§6) run without trained probes. The `concordance` / `ct_gap` groups need one **`SpatialCTClassifierNet`**, trained on the classifier slide from §2 (`../data/abca_1.026_clf.pkl`) in the shared whitened **X_pca(50)** basis. This pair has **20** cell types, so override `data.n_classes=20`; `data.n_pcs=50` sets the net input dimension.

The `recon/*` group needs a second checkpoint: a masked-centroid expression regressor. It reuses `SpatialCTClassifierNet` with `output_dim = n_pcs`, but its dataset target is `target="expr"`, so it predicts the centroid expression from its KNN neighbours.


In [ ]:
%%bash
# Runs on the job's GPU (kernel is inside the SLURM allocation). `cd ..` -> repo root so $PWD and the
# relative out-dirs match a terminal there. Both probes train in the shared whitened X_pca(50), so
# data.n_pcs=50 sets the nets' input_dim (and the regressor's output_dim) to match.
cd ..

# Cell-type classifier checkpoint (monitors val/f1) -> ckpts/clf_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 data.n_pcs=50 \
    logger=tensorboard hydra.run.dir=outputs/clf_gene_train

# Masked-centroid expression regressor checkpoint (monitors val/mse) -> ckpts/recon_spatial.ckpt.
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial_regressor \
    data.datamodule.data_fp=$PWD/data/abca_1.026_clf.pkl data.n_classes=20 data.n_pcs=50 \
    logger=tensorboard hydra.run.dir=outputs/recon_train


In [ ]:
%%bash
cd ../
mkdir -p ckpts
cp "$(ls -t outputs/clf_gene_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/clf_gene.ckpt
cp "$(ls -t outputs/recon_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/recon_spatial.ckpt
echo wrote ckpts/clf_gene.ckpt
echo wrote ckpts/recon_spatial.ckpt


## 4. Train OT-CFM on the shared basis

### 4a. Build `shared_pca.npz` from `abca_pair.pkl`

Export the §2 shared PCA recipe into the format `fm_mnist` can replay. The export goes
through `pp.Basis` (`Basis.from_dataclass(pair).to_fm_npz(...)`) — the single definition of
the shared space — so training and evaluation are guaranteed to use the same whitened
`X_pca(50)` basis (and the same standardised coordinate frame) as NicheFlow.

### 4b. Train OT-CFM on the shared basis

After exporting the basis, retrain OT-CFM with `--pca_stats`. The generator keeps the sampled
expression in PCA space instead of inverting it back to raw genes.


In [ ]:
%%bash
# Export §2's shared PCA as an fm_mnist load_spatial_pca stats .npz (CPU, light; eval venv).
cd ..
uv run python -m paired_slides_eval.adapters.otcfm_export \
    --pair data/abca_pair.pkl --out data/shared_pca.npz


In [ ]:
%%bash
# Retrain OT-CFM IN the shared 50-d basis (gene-only + naive-spatial). Uses the root venv
# (.venv: torch + torchcfm + scanpy + sklearn); train_cfm_spatial.py self-adds fm to sys.path.
# Needs a GPU allocation. Keep your usual hyperparameters (--steps/--batch/--lr/--coupling/…);
# --n_pcs/--whiten/--target_sum are ignored once --pca_stats is given (dim auto = 50, or 52 with coords).
cd ../..                                     # -> generative_modeling/
PY=.venv/bin/python
DATA=nicheflow_mba/data
STATS=eval_metrics/data/shared_pca.npz

# gene-only  (-> feeds artifacts/otcfm_1025)
$PY fm_mnist/scripts/train_cfm_spatial.py \
    --data $DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    --pca_stats $STATS \
    --out fm_mnist/outputs/cfm_shared50_1025

# naive-spatial  (-> feeds artifacts/otcfm_spatial_1025)
$PY fm_mnist/scripts/train_cfm_spatial.py \
    --data $DATA/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    --pca_stats $STATS --spatial_key spatial \
    --out fm_mnist/outputs/cfm_spatial_shared50_1025


## 5. Train NicheFlow VFM and CFM

Train NicheFlow against the same source→target pair used in §2. NicheFlow's preprocessing already produces whitened `X_pca(50)` expression, standardised 2D coordinates, and fixed target-centroid ground truth (`gt_x/gt_pos/gt_ct`) for generated niches.

The exact command can vary with the external `nicheflow_mba` checkout, so keep the project-specific training entrypoint here. After training, use the generation cells in §1 to write `artifacts/nicheflow_cfm_unaligned/generated.h5ad` and `artifacts/nicheflow_vfm_unaligned/generated.h5ad`.


In [ ]:
%%bash
# OPTIONAL — train NicheFlow CFM/VFM checkpoints in the external NicheFlow environment.
# Replace the command/module names with the current training entrypoint in ../nicheflow_mba.
cd ../../nicheflow_mba
# .venv/bin/python -m nicheflow.train variant=cfm \
#     data.source=data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
#     data.target=data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
#     trainer.default_root_dir=ckpts/NicheFlow_CFM_ABCA
# .venv/bin/python -m nicheflow.train variant=vfm \
#     data.source=data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
#     data.target=data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
#     trainer.default_root_dir=ckpts/NicheFlow_VFM_ABCA


## 6. Run evaluate and generate tables and UMAP

This is the same `python -m paired_slides_eval.evaluate` command you'd run in the terminal. Point every model at the **same** `data/abca_pair.pkl`; reconciliation into the shared space is automatic.

- **Expression** — already-reduced shared-PCA cells pass through; gene-space cells would project through the pickle's `SharedGenePCA`.
- **Coordinates** — `--coords auto` detects raw vs already-standardised coords and maps them into the target frame.
- **Flat-slide classifier pairing** — default `--flat_pairing fixed_target_ot` uses `subsampled_timepoint_idx` target centroids and OT-assigns them to generated cells, matching the NicheFlow target frame.

`--classifier` enables `ct/*`; `--ct_real_reference fixed` makes `ct/acc_real` one model-independent constant across the table. `--regressor` enables `recon/*`.


### 6a. Full metric suite → per-model CSV


In [ ]:
%%bash
cd ..
CKPT=ckpts/clf_gene.ckpt
RECON=ckpts/recon_spatial.ckpt
CLF=""; [ -f "$CKPT" ] && CLF="--classifier $CKPT --ct_real_reference fixed"
REG=""; [ -f "$RECON" ] && REG="--regressor $RECON"
[ -z "$CLF" ] && echo "no classifier at $CKPT -> ct/* groups will skip"
[ -z "$REG" ] && echo "no regressor at $RECON -> recon/* group will skip"
METRIC_GROUPS="distribution c2st c2st_nn moran concordance ct_gap recon"
for m in otcfm_shared50_1025 otcfm_spatial_shared50_1025 nicheflow_cfm_unaligned nicheflow_vfm_unaligned; do
    echo "===== $m ====="
    uv run python -m paired_slides_eval.evaluate \
        --target data/abca_pair.pkl \
        --generated artifacts/$m/generated.h5ad \
        --ct_key class --flat_pairing fixed_target_ot --groups $METRIC_GROUPS $CLF $REG \
        --out reports/$m/metrics_npc50_new.csv
done


**Optional** — collate the per-model `metrics.csv` files the CLI just wrote into one wide
comparison table (`reports/model_comparison.csv`)


In [ ]:
import glob
import os

import pandas as pd

cols = {}
for csv_path in sorted(glob.glob("../reports/*/metrics_npc50_new.csv")):
    name = os.path.basename(os.path.dirname(csv_path))
    cols[name] = pd.read_csv(csv_path, index_col="metric")["value"]
table = pd.DataFrame(cols).sort_index()
table.to_csv("../reports/model_comparison_shared50_new.csv")
print("wrote ../reports/model_comparison_shared50_new.csv")
table


### 6d. The same table straight from Python (`me.compare`)

`paired_slides_eval.me.compare` runs `evaluate_files` for each model and concatenates the
columns into one metrics × models `DataFrame` — no bash loop or intermediate CSVs. Space +
coordinate reconciliation happen inside `evaluate_files` exactly as in the CLI above.


In [ ]:
import os

import paired_slides_eval as pse

CLF = "../ckpts/clf_gene.ckpt"
RECON = "../ckpts/recon_spatial.ckpt"
GROUPS = ("distribution", "c2st", "c2st_nn", "moran", "concordance", "ct_gap", "recon")
kw = dict(ct_key="class", groups=GROUPS)
if os.path.exists(CLF):
    kw.update(classifier=CLF, ct_real_reference="fixed")
if os.path.exists(RECON):
    kw.update(regressor=RECON)

# ARTIFACTS + PAIR_PKL come from cells above; each model is scored against the same pair pickle.
table = pse.me.compare(
    {name: (PAIR_PKL, path) for name, path in ARTIFACTS.items()},
    from_files=True, **kw,
)
table.to_csv("../reports/model_comparison_shared50.csv")
table


### 6b. A single model / subset of groups → print

Drop `--out` and the CLI just prints the metrics plus notes. Available groups: `distribution`, `c2st`, `c2st_nn`, `moran`, and — with `--classifier` / `--regressor` — `concordance`, `ct_gap`, `recon`.


In [ ]:
%%bash
cd ..
python -m paired_slides_eval.evaluate \
    --target data/abca_pair.pkl \
    --generated artifacts/otcfm_spatial_shared50_1025/generated.h5ad \
    --classifier ckpts/clf_gene.ckpt --ct_real_reference fixed --flat_pairing fixed_target_ot \
    --regressor ckpts/recon_spatial.ckpt \
    --ct_key class --groups distribution c2st c2st_nn moran recon


### 6c. UMAP overlay — generated vs target

A qualitative companion to the distributional metrics. One UMAP is fit once on the target slide's shared whitened `X_pca(50)`, and each model's generated cells are projected through that frozen embedding so all panels share one coordinate frame.


In [ ]:
from paired_slides_eval.viz import umap_compare

MODELS = [
    ("otcfm",          "../artifacts/otcfm_shared50_1025/generated.h5ad"),
    ("otcfm_spatial",  "../artifacts/otcfm_spatial_shared50_1025/generated.h5ad"),
    ("nicheflow_cfm",  "../artifacts/nicheflow_cfm_unaligned/generated.h5ad"),
    ("nicheflow_vfm",  "../artifacts/nicheflow_vfm_unaligned/generated.h5ad"),
]

fig = umap_compare(
    "../data/abca_pair.pkl", MODELS,
    out_path="../reports/umap_compare.png",
    seed=0, point_size=3, alpha=0.5,   # lower point_size/alpha if the "all cells" plot is too dense
)
fig  # displays inline
